# Automating Your Data Pipeline

##  Learning Outcomes

By the end of this video, you'll be able to:

<ul>
    <li>Build an automated data pipeline</li>
    <li>Implement error handling and monitoring</li>
    <li>Integrate with AWS S3</li>
</ul>

## Setup and Configuration

In [1]:
import pandas as pd
import numpy as np
from datetime import datetime
import boto3
import logging
from io import StringIO
from botocore.exceptions import ClientError

In [2]:
logging.basicConfig(level=logging.INFO,
                   format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

In [3]:
s3_client = boto3.client('s3')
BUCKET_NAME = 'ticketwise-pipeline'

## Data Ingestion

In [4]:
def ingest_ticket_data(bucket_name, file_key):
    """
    Ingest ticket data from S3 with validation
    """
    logger.info(f"Starting ingestion from S3 bucket: {bucket_name}, key: {file_key}")
    
    try:
               
        # Download object from S3
        response = s3_client.get_object(Bucket=bucket_name, Key=file_key)
        
        # Read content into pandas DataFrame
        content = response['Body'].read().decode('utf-8')
        df = pd.read_csv(StringIO(content))
        
        # Validate required columns
        required_columns = ['submission_time', 'channel', 'issue_type']
        missing_columns = [col for col in required_columns if col not in df.columns]
        
        if missing_columns:
            raise ValueError(f"Missing required columns: {missing_columns}")
        
        logger.info(f"Successfully ingested {len(df)} records")
        return df
    
    except ClientError as e:
        logger.error(f"AWS error during ingestion: {e}")
        raise
    except Exception as e:
        logger.error(f"Ingestion failed: {str(e)}")
        raise

df = ingest_ticket_data("ticketwise-pipeline", "ticketwise_dataset.csv")
print(df.head())

2025-09-01 22:05:23,030 - INFO - Starting ingestion from S3 bucket: ticketwise-pipeline, key: ticketwise_dataset.csv
2025-09-01 22:05:23,177 - INFO - Successfully ingested 5000 records


         company_name   channel      submission_time  attachments  \
0       InsightlySoft  web_form  2025-06-18 20:01:37            2   
1  GlobalFinance Corp  web_form  2025-06-20 13:01:37            0   
2       InsightlySoft  web_form   2025-07-12 8:01:37            1   
3       InsightlySoft  web_form   2025-07-10 8:01:37            0   
4       InsightlySoft  web_form   2025-07-01 0:01:37            1   

   previous_interactions  day_of_week subscription_tier  contract_value  \
0                      3            4        Enterprise          211.17   
1                      3            6             Basic         2827.28   
2                      0            7             Basic         5845.15   
3                      1            5        Enterprise         4975.12   
4                      5            3               Pro         5815.52   

   account_age_months  open_tickets  ...               subscribed_modules  \
0                  21             2  ...                B

## Data Processing

In [5]:
def process_ticket_data(df):
    """
    Clean and transform ticket data
    """
    logger.info("Starting ticket processing")
    
    try:
        # Convert submission_time to datetime
        df['submission_time'] = pd.to_datetime(df['submission_time'])
        
        # Standardize channel names
        df['channel'] = df['channel'].str.lower().str.strip()
        
        # Calculate derived features
        df['hour_of_day'] = df['submission_time'].dt.hour
        df['is_business_hours'] = df['hour_of_day'].between(9, 17)
        
        # Handle missing values
        df['issue_type'] = df['issue_type'].fillna('Other')
        
        logger.info("Processing completed successfully")
        return df
        
    except Exception as e:
        logger.error(f"Processing failed: {str(e)}")
        raise

df_processed = process_ticket_data(df)
df_processed.head()

2025-09-01 22:07:16,420 - INFO - Starting ticket processing
2025-09-01 22:07:16,438 - INFO - Processing completed successfully


,company_name,channel,submission_time,attachments,previous_interactions,day_of_week,subscription_tier,contract_value,account_age_months,open_tickets,...,avg_resolution_tier,issue_type,product_area,linked_ticket_count,is_vip,self_declared_p1,is_urgent,resolution_time,hour_of_day,is_business_hours
0,InsightlySoft,web_form,2025-06-18 20:01:37,2,3,4,Enterprise,211.17,21,2,...,25.468611,Performance,Security,0,0,0,1,73.8,20,False
1,GlobalFinance Corp,web_form,2025-06-20 13:01:37,0,3,6,Basic,2827.28,20,1,...,42.330154,Feature Request,Collaboration,0,0,0,0,32.6,13,True
2,InsightlySoft,web_form,2025-07-12 08:01:37,1,0,7,Basic,5845.15,38,0,...,29.892775,Bug,Collaboration,0,0,0,0,52.6,8,False
3,InsightlySoft,web_form,2025-07-10 08:01:37,0,1,5,Enterprise,4975.12,13,1,...,45.318060,How-To,Analytics,0,1,0,1,49.9,8,False
4,InsightlySoft,web_form,2025-07-01 00:01:37,1,5,3,Pro,5815.52,29,2,...,47.794030,Bug,Integration,1,0,0,0,48.0,0,False


## Storage Integration

In [6]:
def store_processed_data(df, bucket, file_name):
    """
    Store processed data in S3
    """
    logger.info(f"Storing data to S3: {bucket}/{file_name}")
    
    try:
        # Convert DataFrame to CSV buffer
        csv_buffer = df.to_csv(index=False)
        
        # Upload to S3
        s3_client.put_object(
            Bucket=bucket,
            Key=file_name,
            Body=csv_buffer
        )
        
        logger.info("Data successfully stored in S3")
        
    except ClientError as e:
        logger.error(f"S3 operation failed: {str(e)}")
        raise

store_processed_data(df_processed, BUCKET_NAME, "ticketwise_dataset_processed.csv")


2025-09-01 22:09:07,048 - INFO - Storing data to S3: ticketwise-pipeline/ticketwise_dataset_processed.csv
2025-09-01 22:09:07,259 - INFO - Data successfully stored in S3


## Pipeline Integration

In [ ]:
def run_pipeline(bucket_name, file_key):
    """
    Execute complete pipeline
    """
    try:
        # Step 1: Ingest
        raw_data = ingest_ticket_data(bucket_name, file_key)
        
        # Step 2: Process
        processed_data = process_ticket_data(raw_data)
        
        # Step 3: Store
        today = datetime.now().strftime('%Y-%m-%d')
        store_processed_data(
            processed_data,
            BUCKET_NAME,
            f'ticketwise_dataset_processed_{today}.csv'
        )
        
        logger.info("Pipeline completed successfully")
        
    except Exception as e:
        logger.error(f"Pipeline failed: {str(e)}")
        raise
        
run_pipeline("ticketwise-pipeline", "ticketwise_dataset.csv")

## Recap


In this video, we covered:

- Pipeline component creation
  
- Error handling implementation

- AWS S3 integration

- Logging and monitoring setup
